# 🔥 Notebook 6: Hot Keys and Advanced Patterns

When one key receives disproportionate traffic, it becomes a bottleneck. Learn to detect and handle hot keys.

## Learning Objectives

By the end of this notebook, you'll understand:
- What causes hot keys
- Detection strategies
- Fixed-k key splitting
- Dynamic key splitting (driven by a *rate*, not a running total)
- Online resharding without downtime
- Real-world examples

In [ ]:
import redis
import random
import time
from collections import defaultdict
from typing import Dict, List, Optional

# Seeded so every distribution printed in this notebook is reproducible.
random.seed(42)

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.flushall()

print("✅ Connected to Redis!")
print("📊 Open RedisInsight: http://localhost:5540")

## 🔥 The Hot Key Problem

In [ ]:
print("🔥 The Hot Key Problem")
print("=" * 60)
print("""
SCENARIO: Celebrity tweet during Super Bowl
─────────────────────────────────────────────────────────────

Taylor Swift tweets: "Go Chiefs! 🏈"

┌────────────────────────────────────────────────────────────┐
│    1 million likes in 60 seconds                          │
│                                                            │
│    All 1M writes go to: tweet_id = 12345                  │
│                                                            │
│    Shard 3 (where 12345 lives): 💥 OVERLOADED!            │
│    Other shards: 😴 Idle                                  │
└────────────────────────────────────────────────────────────┘

PROBLEM:
• Sharding doesn't help - all traffic goes to ONE shard
• That shard becomes bottleneck
• Other shards sit idle

HOT KEY CAUSES:
• Celebrity posts (millions of followers)
• Flash sales (one product_id)
• Breaking news (one article_id)
• Live events (one event_id)
""")

In [ ]:
print("🔬 Simulating Hot Key Problem")
print("=" * 60)

shard_counts = defaultdict(int)
NUM_SHARDS = 10

def get_shard(key: int) -> int:
    return key % NUM_SHARDS

regular_posts = list(range(1, 10001))
celebrity_post = 12345

print("\n📝 Normal traffic: 10,000 likes across 10,000 posts")
for post_id in regular_posts:
    shard_counts[get_shard(post_id)] += 1

print("📊 Shard distribution (normal):")
for shard in range(NUM_SHARDS):
    bar = "█" * (shard_counts[shard] // 100)
    print(f"   Shard {shard}: {shard_counts[shard]:>5} writes {bar}")

# Snapshot the baseline before reusing the counter for the hot-key run.
normal_counts = {s: shard_counts[s] for s in range(NUM_SHARDS)}
shard_counts.clear()

print("\n🔥 Hot key traffic: 10,000 likes on ONE celebrity post")
for _ in range(10000):
    shard_counts[get_shard(celebrity_post)] += 1

print("📊 Shard distribution (hot key):")
for shard in range(NUM_SHARDS):
    bar = "█" * (shard_counts[shard] // 100)
    print(f"   Shard {shard}: {shard_counts[shard]:>5} writes {bar}")

hot_counts = {s: shard_counts[s] for s in range(NUM_SHARDS)}


def imbalance(counts: dict) -> float:
    """Hottest shard's load / average shard's load.

    1.0 = perfectly even. NUM_SHARDS = one shard is doing everything, which is
    the worst score the metric can produce.
    """
    return max(counts.values()) / (sum(counts.values()) / NUM_SHARDS)


hot_shard = get_shard(celebrity_post)
print(f"\n❌ All traffic goes to shard {hot_shard} "
      f"({celebrity_post} % {NUM_SHARDS} = {hot_shard})!")
print(f"   Imbalance factor: {imbalance(normal_counts):.1f}x (normal traffic) "
      f"→ {imbalance(hot_counts):.1f}x (hot key)")
print(f"   {imbalance(hot_counts):.0f}x is the worst score {NUM_SHARDS} shards can produce:")
print(f"   {NUM_SHARDS - 1} of {NUM_SHARDS} machines idle while one melts. Adding shard")
print(f"   number {NUM_SHARDS + 1} does not help -- the key still hashes to one place.")

# The contrast is the entire lesson. Both ends have to keep holding.
assert imbalance(normal_counts) == 1.0, (
    f"10,000 sequential ids over {NUM_SHARDS} shards is exactly even, got "
    f"{imbalance(normal_counts):.2f}x"
)
assert imbalance(hot_counts) == NUM_SHARDS, (
    f"a single hot key must pin 100% of writes to one shard, got "
    f"{imbalance(hot_counts):.2f}x from {hot_counts}"
)

## 🔍 Hot Key Detection

In [ ]:
class HotKeyDetector:
    """Flags any key taking more than threshold_ratio of all writes so far.

    ⚠️ Two simplifications worth naming, because they are the difference
    between this demo and a real detector:

    1. The counts are CUMULATIVE since process start, and `check_every` is a
       check cadence, not a sliding window. A key that was viral this morning
       keeps its share of the running total all day.
    2. `hot_keys` never cools down. Once flagged, always flagged -- so the
       split from the next section would live forever.

    Production detectors keep a decaying or sliding-window count (often a
    count-min sketch, so the memory cost does not scale with key cardinality)
    precisely so a key can go cold again and the split can be undone.
    """

    def __init__(self, check_every: int = 100, threshold_ratio: float = 0.1):
        self.check_every = check_every
        self.threshold_ratio = threshold_ratio
        self.counts: Dict[str, int] = defaultdict(int)
        self.total_writes = 0
        self.hot_keys: set = set()
    
    def record(self, key: str):
        self.counts[key] += 1
        self.total_writes += 1
        
        if self.total_writes % self.check_every == 0:
            self._detect_hot_keys()
    
    def _detect_hot_keys(self):
        threshold = self.total_writes * self.threshold_ratio
        for key, count in self.counts.items():
            if count > threshold:
                if key not in self.hot_keys:
                    self.hot_keys.add(key)
                    print(f"   🔥 HOT KEY DETECTED: {key} ({count} writes, {count/self.total_writes*100:.1f}%)")
    
    def is_hot(self, key: str) -> bool:
        return key in self.hot_keys

print("🔍 Hot Key Detection Demo")
print("=" * 60)

detector = HotKeyDetector(check_every=100, threshold_ratio=0.1)

print("\n📝 Writing traffic (mostly normal, some hot)...")
for i in range(1000):
    if random.random() < 0.3:
        detector.record("celebrity_post_123")
    else:
        detector.record(f"normal_post_{random.randint(1, 100)}")

print(f"\n📊 Summary:")
print(f"   Total writes: {detector.total_writes}")
print(f"   Hot keys found: {detector.hot_keys}")

# ~30% of traffic on one key must trip a 10% threshold, and the ~0.7%-each
# ordinary keys must not. If either side moves, the detector is either blind
# or crying wolf.
assert "celebrity_post_123" in detector.hot_keys, (
    f"a key taking ~30% of writes must be flagged, hot_keys={detector.hot_keys}"
)
assert not any(k.startswith("normal_post_") for k in detector.hot_keys), (
    f"ordinary keys (~0.7% each) must not be flagged: {detector.hot_keys}"
)


## 🔀 Fixed-K Key Splitting

In [ ]:
print("🔀 Fixed-K Key Splitting")
print("=" * 60)
print("""
IDEA: Split hot key into K sub-keys, spread across shards

BEFORE:
─────────────────────────────────────────────────────────────
    tweet_12345 → Shard 5 (ALL writes here!)

AFTER (K=4):
─────────────────────────────────────────────────────────────
    tweet_12345_0 → Shard 2
    tweet_12345_1 → Shard 7
    tweet_12345_2 → Shard 1  
    tweet_12345_3 → Shard 9

WRITES:
    Writer picks random sub-key: tweet_12345_{random(0,3)}
    Spreads load across 4 shards!

READS:
    Total likes = sum(tweet_12345_0 ... tweet_12345_3)
    Query all K sub-keys and sum
""")

In [ ]:
class FixedKSplitter:
    def __init__(self, k: int = 4):
        self.k = k
        self.hot_keys: set = set()
    
    def mark_hot(self, key: str):
        self.hot_keys.add(key)
    
    def get_write_key(self, key: str) -> str:
        if key in self.hot_keys:
            suffix = random.randint(0, self.k - 1)
            return f"{key}_{suffix}"
        return key
    
    def get_read_keys(self, key: str) -> List[str]:
        if key in self.hot_keys:
            return [f"{key}_{i}" for i in range(self.k)]
        return [key]
    
    def increment(self, key: str):
        write_key = self.get_write_key(key)
        r.incr(write_key)
        return write_key
    
    def get_total(self, key: str) -> int:
        read_keys = self.get_read_keys(key)
        total = 0
        for rk in read_keys:
            val = r.get(rk)
            if val:
                total += int(val)
        return total

print("🔬 Fixed-K Splitting Demo")
print("=" * 60)

splitter = FixedKSplitter(k=4)
splitter.mark_hot("celebrity_tweet")

# Clear counters from any previous run of this cell -- otherwise re-executing
# it doubles the total and the "1,000 likes" claim below quietly stops being true.
r.delete(*splitter.get_read_keys("celebrity_tweet"))

print("\n📝 Writing 1000 likes to hot key (split into 4)...")
write_distribution = defaultdict(int)
for _ in range(1000):
    written_key = splitter.increment("celebrity_tweet")
    write_distribution[written_key] += 1

print("\n📊 Write Distribution:")
for key in sorted(write_distribution.keys()):
    count = write_distribution[key]
    bar = "█" * (count // 25)
    print(f"   {key}: {count:>4} writes {bar}")

print(f"\n📖 Reading total:")
total = splitter.get_total("celebrity_tweet")
print(f"   Total likes: {total}")

hottest = max(write_distribution.values()) / (1000 / splitter.k)
print(f"\n✅ Hot key spread across {splitter.k} sub-keys!")
print(f"   Imbalance across sub-keys: {hottest:.2f}x "
      f"(1.00 = perfect, {splitter.k}.00 = no spreading at all)")
print(f"   Cost: reading the total is now {splitter.k} GETs instead of 1.")

# Splitting is only correct if the fan-out read finds every write back.
assert total == 1000, (
    f"key splitting must not lose writes: wrote 1,000, read back {total}"
)
# And it is only useful if the writes actually spread.
assert hottest < 1.3, (
    f"a random suffix should spread writes to within ~30% of even, got "
    f"{dict(write_distribution)}"
)

## 📈 Dynamic Key Splitting

In [ ]:
print("📈 Dynamic Key Splitting")
print("=" * 60)
print("""
PROBLEM: Fixed K is wasteful for varying traffic levels
─────────────────────────────────────────────────────────────

• Low traffic: K=4 means 4 reads for every query (overkill)
• High traffic: K=4 might not be enough

SOLUTION: Dynamically adjust K based on traffic
─────────────────────────────────────────────────────────────

Strategy:
1. Start with K=1 (no splitting)
2. Monitor write rate per key
3. If rate > threshold, double K
4. Migrate data to new sub-keys

Example:
    Normal: celebrity_tweet (K=1)
    Hot:    celebrity_tweet_0, celebrity_tweet_1 (K=2)
    Viral:  celebrity_tweet_0..3 (K=4)
""")

In [ ]:
class DynamicSplitter:
    """Adjusts K from the key's CURRENT write rate.

    The word "rate" is doing real work here. A naive implementation compares a
    cumulative write counter against a threshold -- but a cumulative counter
    only ever goes up, so K doubles forever. A key doing a sleepy 1 write/sec
    would end up split 64 ways after a day, paying 64 reads per lookup for
    traffic a single key could serve. The count has to be measured over a
    window and reset, so the value can come back down.
    """

    def __init__(self, rate_threshold: float = 200.0, window_seconds: float = 1.0):
        # Writes/sec *per sub-key* we are willing to point at one shard.
        self.rate_threshold = rate_threshold
        self.window_seconds = window_seconds
        self.key_k: dict = defaultdict(lambda: 1)
        # Reads must fan out over the largest K the key has EVER had. Lowering
        # K stops new writes going to the high suffixes; it does not move the
        # counts already sitting in them.
        self.read_k: dict = defaultdict(lambda: 1)
        self.window_writes: dict = defaultdict(int)
        self.window_start: dict = {}

    def increment(self, key: str, now: float):
        """Record one write for `key` at simulated time `now` (seconds)."""
        if key not in self.window_start:
            self.window_start[key] = now

        # Always write to a suffixed sub-key, even at K=1, so counts written
        # before a split stay findable afterwards -- no migration step.
        k = self.key_k[key]
        r.incr(f"{key}_{random.randint(0, k - 1)}")
        self.window_writes[key] += 1

        if now - self.window_start[key] >= self.window_seconds:
            self._reconsider(key, now)
        return k

    def _reconsider(self, key: str, now: float):
        elapsed = now - self.window_start[key]
        rate = self.window_writes[key] / elapsed      # writes/sec for the key
        current_k = self.key_k[key]
        per_subkey = rate / current_k                 # writes/sec per shard

        if per_subkey > self.rate_threshold:
            new_k = current_k * 2
            self.key_k[key] = new_k
            self.read_k[key] = max(self.read_k[key], new_k)
            print(f"   📈 t={now:5.1f}s  rate={rate:>7,.0f}/s  "
                  f"split {key}: K={current_k} → K={new_k}")
        # Hysteresis: only merge back if HALF the current K would still leave
        # us comfortably (2x) under the threshold. Without that gap, a key
        # sitting near the line would flap between K and 2K every window.
        elif current_k > 1 and rate / (current_k // 2) < self.rate_threshold * 0.5:
            new_k = current_k // 2
            self.key_k[key] = new_k
            print(f"   📉 t={now:5.1f}s  rate={rate:>7,.0f}/s  "
                  f"merge {key}: K={current_k} → K={new_k}")

        self.window_writes[key] = 0
        self.window_start[key] = now

    def get_total(self, key: str) -> int:
        # Fan out over read_k, not key_k: a merge lowered the write fan-out,
        # but the counts written while K was larger are still up there.
        total = 0
        for i in range(self.read_k[key]):
            val = r.get(f"{key}_{i}")
            if val:
                total += int(val)
        return total


print("🔬 Dynamic Splitting Demo")
print("=" * 60)

r.flushall()
random.seed(7)
dynamic = DynamicSplitter(rate_threshold=200.0, window_seconds=1.0)

# A simulated clock. Each phase declares how many writes arrive AND how fast,
# so "traffic increases over time" is a property of the input rather than an
# artefact of a counter that only goes up. Phase 4 is the important one: the
# traffic genuinely stops, and K has to notice.
phases = [
    ("1. Normal traffic",   600,    100.0),   #    100 writes/sec for 6s
    ("2. Going viral",    4_000,  1_000.0),   #  1,000 writes/sec for 4s
    ("3. Super viral",   25_000, 10_000.0),   # 10,000 writes/sec for 2.5s
    ("4. Cooling off",      600,     40.0),   #     40 writes/sec for 15s
]

now = 0.0
total_written = 0
k_after_phase = []

for label, count, rate in phases:
    print(f"\n   {label}: {count:,} writes at {rate:,.0f}/sec")
    for _ in range(count):
        now += 1.0 / rate
        dynamic.increment("viral_post", now)
        total_written += 1
    k_after_phase.append(dynamic.key_k["viral_post"])
    print(f"      K after phase: {dynamic.key_k['viral_post']}  "
          f"(reads still fan out over {dynamic.read_k['viral_post']})")

final_k = dynamic.key_k["viral_post"]
peak_k = dynamic.read_k["viral_post"]
recorded = dynamic.get_total("viral_post")

print(f"\n📊 Writes issued: {total_written:,}   read back: {recorded:,}")
print(f"   K by phase: {k_after_phase}   peak K: {peak_k}   final K: {final_k}")

# 1. Nothing may be lost or double-counted across every split and merge.
assert recorded == total_written, (
    f"splitting and merging must be lossless: wrote {total_written:,}, "
    f"read back {recorded:,}"
)
# 2. K must NOT grow while the rate is below threshold. This is the assertion
#    that catches the classic bug: if the splitter is driven by a cumulative
#    counter instead of a rate, phase 1 alone will already have split the key.
assert k_after_phase[0] == 1, (
    f"K must stay at 1 while the write rate is under threshold; it reached "
    f"{k_after_phase[0]}, which means the splitter is reacting to total writes "
    f"rather than to a rate"
)
# 3. K must track the rate up...
assert k_after_phase[2] > k_after_phase[1] > k_after_phase[0], (
    f"K must rise as the rate rises, got {k_after_phase}"
)
# 4. ...and back down when the traffic goes away.
assert k_after_phase[3] < k_after_phase[2], (
    f"K must fall again once traffic cools, got {k_after_phase}"
)
assert final_k < peak_k, (
    f"final K ({final_k}) should be below the peak ({peak_k})"
)

print(f"""
💡 What this actually costs:

   • Reads still fan out to {peak_k} sub-keys even though writes are back down to
     {final_k}. Lowering K is free; lowering the READ fan-out is not -- you have to
     migrate the counts in suffixes {final_k}..{peak_k - 1} down into the low ones and
     only then lower read_k. That migration is the resharding cost in
     miniature, and it is why nobody reshards casually.
   • While K is changing there is no single row holding the truth. Every read
     is a scatter-gather across {peak_k} keys, and any per-row consistency
     guarantee you had is gone -- you are summing values read at slightly
     different instants.
   • The counter is now approximate by construction. That is fine for likes.
     It is not fine for account balances.
""")


## 🔁 Resharding Without Downtime

Splitting a key, adding a shard, changing the hash — all of them mean the same
operation: **data has to move while the system keeps taking writes.** You
cannot stop the world, and you cannot copy an inconsistent snapshot.

The naive version fails in a way that is easy to miss, because it fails
*quietly*. Read everything from the old shard, write it to the new one, flip
reads over. Any write that lands between the read and the flip exists only on
the old shard and disappears at cutover. No error, no exception, no leftover —
just a number that is silently wrong forever.

The standard fix inverts the order: start double-writing **before** you copy
anything, so the new shard is never behind, only incomplete. Then backfill the
history underneath the live traffic.

```
1. DOUBLE-WRITE  every write goes to old AND new. Old is still authoritative.
2. BACKFILL      copy history to new, in throttled chunks, under live load.
3. VERIFY        read both, compare, alert on drift ("dark reads").
4. CUT OVER      point reads at new. Then, and only then, stop double-writing.
```

Every step is individually reversible, which is the entire point — at any
moment you can turn reads back to the old shard and lose nothing.

Below we run the same one-write-mid-migration scenario through both
approaches.

In [ ]:
print("🔁 Resharding: naive copy vs double-write + backfill")
print("=" * 60)


def reset_shards():
    """100 posts on the old shard, post_7 sitting at 7 likes."""
    r.delete("old_shard", "new_shard")
    r.hset("old_shard", mapping={f"post_{i}": i for i in range(100)})


def old_value(field="post_7"):
    return int(r.hget("old_shard", field))


def new_value(field="post_7"):
    raw = r.hget("new_shard", field)
    return int(raw) if raw is not None else None


# --- Attempt 1: copy, then cut over ---------------------------------------
reset_shards()

snapshot = r.hgetall("old_shard")                      # 1. read everything
r.hset("new_shard", mapping=snapshot)                  # 2. write it to the new shard
r.hincrby("old_shard", "post_7", 1)                    # 3. 💥 a like lands HERE,
                                                       #    after the copy, before cutover.
                                                       #    It only knows about old_shard.
# 4. cut over: readers now go to new_shard
print("\n1️⃣ Naive copy, one like arrives between the copy and the cutover:")
print(f"   old_shard post_7 = {old_value()}")
print(f"   new_shard post_7 = {new_value()}")
print(f"   ❌ the like vanished at cutover -- no error, no retry, no leftover")

naive_new = new_value()
naive_old = old_value()

# --- Attempt 2: double-write, backfill, then cut over ---------------------
reset_shards()
DOUBLE_WRITE = True


def like(field="post_7"):
    """A production write during the migration window."""
    r.hincrby("old_shard", field, 1)          # old shard stays authoritative
    if DOUBLE_WRITE:
        r.hincrby("new_shard", field, 1)      # ...and the new one sees it live


# 1. Double-writes go on FIRST. new_shard starts empty but is never stale.
# 2. Backfill the history. old_shard is the source of truth for everything
#    that happened before the switch, so copying its current value over is
#    idempotent -- re-running the backfill is safe.
for field, value in r.hgetall("old_shard").items():
    r.hset("new_shard", field, int(value))

# 3. The SAME like, at the SAME moment in the timeline as attempt 1.
like("post_7")

# 4. Verify, then cut over.
print("\n2️⃣ Double-write + backfill, identical timing:")
print(f"   old_shard post_7 = {old_value()}")
print(f"   new_shard post_7 = {new_value()}")
print(f"   ✅ nothing lost -- the write reached the new shard on its own way in")

safe_new = new_value()
safe_old = old_value()

drift = [f for f, v in r.hgetall("old_shard").items() if int(v) != int(r.hget("new_shard", f))]
print(f"   Verification pass: {len(drift)} field(s) drifted between shards")

# Reproduce the failure, then show the fix. Both halves must keep holding.
assert naive_old == 8 and naive_new == 7, (
    f"the naive copy must lose the mid-migration write: old={naive_old}, "
    f"new={naive_new} (expected 8 and 7)"
)
assert safe_old == 8 and safe_new == 8, (
    f"double-writing must carry the mid-migration write across: old={safe_old}, "
    f"new={safe_new} (expected 8 and 8)"
)
assert drift == [], f"shards must agree before cutover, drifted on {drift}"

print("""
💡 What you are paying for that safety, for the whole duration of the migration:

   • Every write costs double -- two shards, two round trips, two failure modes.
     Plan capacity for it, because the migration is exactly when you are least
     able to absorb a surprise.
   • The backfill competes with production traffic for the same disks and the
     same network. Throttle it, chunk it, and make it resumable; a backfill you
     cannot pause is a backfill that will take your site down.
   • Double-writing is not transactional. If the new-shard write fails and the
     old-shard write succeeded, the shards drift -- which is why step 3 is a
     verification pass and not a formality.

   All of which is the real argument of this whole notebook: the cheapest
   resharding is the one you never have to do. Time spent choosing the shard
   key on day one is worth more than any of this machinery.
""")


## 🧪 Quick Quiz

1. **Why doesn't sharding help with hot keys?**

2. **What's the read trade-off of key splitting?**

3. **When would you prefer fixed-K vs dynamic splitting?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why sharding doesn't help:")
print("   - Hot key always maps to SAME shard")
print("   - All traffic concentrates on one node")
print("   - Other shards sit idle")
print()
print("2. Read trade-off of splitting:")
print("   - Must query K sub-keys instead of 1")
print("   - More network calls or parallel queries")
print("   - Sum results = higher read latency")
print()
print("3. Fixed-K vs Dynamic:")
print("   Fixed-K: Simpler, predictable reads")
print("           Good when hot keys are known")
print("   Dynamic: Adapts to changing traffic")
print("           Better for unpredictable spikes")

## 📚 Summary

### Key Takeaways

1. **Hot keys bypass sharding** - All traffic to one shard
2. **Detect early** - Monitor write rates per key
3. **Fixed-K splitting** - Simple, predictable, works for known hot keys
4. **Dynamic splitting** - Adapts to traffic, handles surprises
5. **Trade-off** - More writes per key = more reads to sum
6. **Resharding is never free** - double-write, backfill, verify, cut over;
   the cheapest reshard is the one a good key choice avoided

### 🎉 Pattern Complete!

You've learned the core strategies for scaling writes:
1. **Database optimization** - Bulk inserts, minimal indexes
2. **Sharding** - Distribute across servers
3. **Queues** - Buffer bursts, steady drain
4. **Batching** - Combine writes, aggregate counters
5. **Hot keys** - Split to spread load

### Decision Framework

```
High write volume?
├── Single key hot? → Key splitting
├── Bursty traffic? → Queues + load shedding
├── Many small writes? → Batching + aggregation
└── Uniform load? → Sharding
```